# SoW · baseline 与评估（可读版）

本 notebook 使用普通项目文件，不内嵌 Base64 编码。

## 使用前准备
1. 将本 notebook 上传到 Colab。
2. 下载同目录下的 `SoW_Evaluation_Readable_Project.zip`，第1步会让你选择上传它。
3. 依次运行。免费步骤不调用模型；留出集开关默认关闭。

项目中的 `baseline.py`、`evaluate.py`、`sow.py`、prompt、数据、固定答案和已有开发输出均为可直接阅读的文本文件。Colab左侧“文件”面板可展开查看。

AI代码和prompt仍是原v4.1；已有10个开发结果保持原样。人工评分未完成时不报告整体准确率。这个包是评估阶段材料，不是已完成所有课程交付的最终作业包。


## 1 · 上传明文项目文件（免费）

运行下方代码后，选择 **SoW_Evaluation_Readable_Project.zip**。普通 ZIP 只用于组织文件；所有代码都在可读的 `.py` 文件中。

程序将文件放入 `/content/sow_evaluation_project`；已有结果、人工记录不会被覆盖。如果这个目录已经包含完整项目，会直接使用，无需重复上传。


In [ ]:
from pathlib import Path
import io, zipfile, json, sys, hashlib, importlib, os
from google.colab import files

PROJECT = Path('/content/sow_evaluation_project')
PROJECT.mkdir(parents=True, exist_ok=True)

required_files = [
    'sow.py', 'baseline.py', 'evaluate.py', 'checks.py',
    'prompts/system.txt', 'expected_manifest.json', 'manual_decisions.json',
    'data/dev_inputs.json', 'data/holdout_inputs.json',
    'labels/dev_ground_truth.json', 'labels/holdout_ground_truth.json',
    'saved_dev_ai/inputs_used.json', 'saved_dev_ai/summary.json',
] + [f'saved_dev_ai/DEV{i:02d}.json' for i in range(1, 11)]

if not all((PROJECT / name).is_file() for name in required_files):
    print('请选择 SoW_Evaluation_Readable_Project.zip')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('请只选择一个项目 ZIP 文件。')
    zip_bytes = next(iter(uploaded.values()))
    prefix = 'sow_evaluation_project/'
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as archive:
        for item in archive.infolist():
            if item.is_dir() or not item.filename.startswith(prefix):
                continue
            relative_name = item.filename[len(prefix):]
            target = (PROJECT / relative_name).resolve()
            if not target.is_relative_to(PROJECT.resolve()):
                raise ValueError('ZIP 中存在非法路径。')
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(item.filename))
    del uploaded, zip_bytes

missing = [name for name in required_files if not (PROJECT / name).is_file()]
if missing:
    raise ValueError('项目文件不完整：' + ', '.join(missing))
print('项目已准备。可从左侧文件面板打开 sow_evaluation_project 查看代码和数据。')
print('本步骤没有调用 API，也不会覆盖已有结果。')


请选择 SoW_Evaluation_Readable_Project.zip


Saving SoW_Evaluation_Readable_Project.zip to SoW_Evaluation_Readable_Project.zip
项目已准备。可从左侧文件面板打开 sow_evaluation_project 查看代码和数据。
本步骤没有调用 API，也不会覆盖已有结果。


## 2 · 免费自检与版本确认
自检阻断网络。没有通过时，不继续付费步骤。

In [ ]:
sys.path.insert(0,str(PROJECT))
for name in ['sow','baseline','evaluate','checks']:sys.modules.pop(name,None)
import sow,baseline,evaluate,checks
EXPECTED=json.loads((PROJECT/'expected_manifest.json').read_text())
MODEL=EXPECTED['model']
def assert_frozen():
    assert MODEL==EXPECTED['model'],'模型已改变，不能算同一冻结版本。'
    for name,digest in EXPECTED['files'].items():
        assert hashlib.sha256((PROJECT/name).read_bytes()).hexdigest()==digest,f'文件已改变：{name}'
    for module in [sow,baseline,evaluate,checks]:
        assert Path(module.__file__).resolve().parent==PROJECT.resolve(),'加载到了其他notebook的模块。'
assert_frozen()
FREE_CHECKS_OK=False
n=checks.run_checks()
FREE_CHECKS_OK=True
from datetime import datetime,timezone
manifest=PROJECT/'freeze_manifest.json'
if not manifest.exists():
    manifest.write_text(json.dumps({**EXPECTED,'frozen_at_utc':datetime.now(timezone.utc).isoformat()},ensure_ascii=False,indent=2))
print(f'{n}项免费自检通过；AI、baseline、评分规则及数据的版本已记录。未调用API。')


.........
----------------------------------------------------------------------
Ran 9 tests in 0.017s

OK


9项免费自检通过；AI、baseline、评分规则及数据的版本已记录。未调用API。


## 3 · 免费运行开发集 baseline，复用 AI 开发结果

本格只运行规则和评估，不调用模型。看到baseline成绩后，不要求它必须低于AI。

In [ ]:
assert FREE_CHECKS_OK
assert_frozen()
baseline.run(PROJECT/'data/dev_inputs.json',PROJECT/'results/dev_baseline')
dev_summary=evaluate.report(PROJECT/'data/dev_inputs.json',PROJECT/'labels/dev_ground_truth.json',
    {'baseline':PROJECT/'results/dev_baseline','ai':PROJECT/'saved_dev_ai'},PROJECT/'reports/dev','dev',PROJECT/'manual_decisions.json')
from IPython.display import display,Markdown,HTML
display(Markdown((PROJECT/'reports/dev/summary.md').read_text()))


# dev 对比结果

行动/类型/问题集合匹配是分项指标，不是总体准确率。整体通过必须按固定规则人工确认。

|系统|案例数|行动匹配|类型匹配|问题集合匹配|应转人工中实际转人工|正常中转人工|API费用USD|人工完成|
|---|---:|---:|---:|---:|---|---|---|---|
|baseline|10|8/10|8/10|6/10|4/5|1/5|0.0|0/10|
|ai|10|9/10|9/10|9/10|5/5|1/5|0.0037278|0/10|

费用为每案例保留的独立响应费用，复用开发结果不是本次新支出。baseline API费用为0，未估算CPU成本。无响应案例保留在分母中。引用存在性不代表语义支持；AI引用文本由程序附上，不能称为模型自主引用准确率。

baseline 完整任务通过数：待人工核验，不报告准确率。

ai 完整任务通过数：待人工核验，不报告准确率。


## 4 · 按同一标准人工核验开发集（免费）

展开每个案例，查看输入、固定答案、系统输出和分项检查。重点核对：金额是谁的、什么口径/期间、币种、是否已收到、证据是否支持、必需比例计算及风险码。允许事实顺序和名称同义变化。

参考已有AI审阅：DEV01缺比例计算；DEV06缺比例计算/股权所得问题码；DEV08分类错误；DEV09基金口径错误。不要把这些案例标成完整成功。DEV07需要核对待付款事实；DEV05状态引用也需检查。所有最终人工判断由你确认。


In [ ]:
display(HTML((PROJECT/'reports/dev/manual_review.html').read_text()))


## 5 · 保存人工结论（免费，可先留待稍后完成）

在下方 `DECISIONS_TO_ADD` 填结论，不改ground truth。例如：
`('dev','ai','DEV01'): {'overall':'FAIL','reason':'缺少要求的股权比例计算'}`

PASS = 满足整套要求；FAIL = 至少一个必要要求不满足；PENDING = 尚未核对。每条结论写一句原因。对baseline和AI一视同仁。未全部确认时，报告不会生成总体准确率。此格可在留出评估后继续填写holdout。


In [ ]:
DECISIONS_TO_ADD={
    ('holdout', 'baseline', 'HOLD01'): {'overall': 'FAIL', 'reason': '把声明的1250000赎回总额提取成250000并标为本金，误报amount_mismatch及转人工；缺少正确的completed状态。'},
    ('holdout', 'baseline', 'HOLD02'): {'overall': 'FAIL', 'reason': '未区分公司估值、合同对价、已付及未付余额，付款状态均unknown；遗漏ownership_proceeds_mismatch和transaction_not_completed。'},
    ('holdout', 'baseline', 'HOLD03'): {'overall': 'FAIL', 'reason': '投资房产错分unknown，未保留净所得口径及completed；漏掉必需currency_not_comparable，只给出泛化的insufficient_information。'},
    ('holdout', 'baseline', 'HOLD04'): {'overall': 'FAIL', 'reason': '两份同一交易结算金额750000与720000冲突，却输出no_issue_detected；遗漏conflicting_documents，付款状态错误为unknown。'},
    ('holdout', 'baseline', 'HOLD05'): {'overall': 'PASS', 'reason': '空输入正确输出unknown、insufficient_input和insufficient_information；事实、证据、计算均为空，未编造信息。'},
    ('holdout', 'ai', 'HOLD01'): {'overall': 'PASS', 'reason': '正确保留1250000 USD赎回总额及本金1000000、收益250000，口径、completed、正常行动及证据正确；固定规则未把额外加法算式列为必需项。'},
    ('holdout', 'ai', 'HOLD02'): {'overall': 'FAIL', 'reason': '虽识别金额差异与未付余额，仍遗漏必需ownership_proceeds_mismatch；将合同2500000状态写成completed，不符合unknown/pending可接受范围，多个状态无引用。'},
    ('holdout', 'ai', 'HOLD03'): {'overall': 'FAIL', 'reason': '将投资房产误作equity_sale；漏掉currency_not_comparable，并把1200000 USD与1500000 SGD直接相减后标成USD，属于无效跨币种计算。'},
    ('holdout', 'ai', 'HOLD04'): {'overall': 'FAIL', 'reason': '正确保留750000/720000两项结算事实并转人工，但仅报告amount_mismatch，遗漏必需conflicting_documents；数值差异不能替代文件冲突分类。'},
    ('holdout', 'ai', 'HOLD05'): {'overall': 'PASS', 'reason': '空输入正确输出unknown、insufficient_input和insufficient_information，事实、证据、计算为空，并说明没有可用财富信息。'},
}

path=PROJECT/'manual_decisions.json'
manual=json.loads(path.read_text())
for (split,system,cid),decision in DECISIONS_TO_ADD.items():
    assert split in manual and system in manual[split] and cid in manual[split][system]
    assert decision.get('overall') in {'PASS','FAIL','PENDING'}
    manual[split][system][cid]=decision
# Preserve a backup before replacing manual decisions.
(PROJECT/'manual_decisions.previous.json').write_text(path.read_text())
path.write_text(json.dumps(manual,ensure_ascii=False,indent=2))
dev_summary=evaluate.report(PROJECT/'data/dev_inputs.json',PROJECT/'labels/dev_ground_truth.json',
    {'baseline':PROJECT/'results/dev_baseline','ai':PROJECT/'saved_dev_ai'},PROJECT/'reports/dev','dev',path)
display(Markdown((PROJECT/'reports/dev/summary.md').read_text()))


# dev 对比结果

行动/类型/问题集合匹配是分项指标，不是总体准确率。整体通过必须按固定规则人工确认。

|系统|案例数|行动匹配|类型匹配|问题集合匹配|应转人工中实际转人工|正常中转人工|API费用USD|人工完成|
|---|---:|---:|---:|---:|---|---|---|---|
|baseline|10|8/10|8/10|6/10|4/5|1/5|0.0|10/10|
|ai|10|9/10|9/10|9/10|5/5|1/5|0.0037278|10/10|

费用为每案例保留的独立响应费用，复用开发结果不是本次新支出。baseline API费用为0，未估算CPU成本。无响应案例保留在分母中。引用存在性不代表语义支持；AI引用文本由程序附上，不能称为模型自主引用准确率。

baseline 完整任务通过数：1/10

ai 完整任务通过数：6/10


## 6 · 冻结版本后运行留出集（默认关闭；最多5次API请求）

准备好后才勾选。程序先验证代码/prompt/baseline/数据/评分文件没有变化，运行免费baseline，再请求5个留出案例。输入只有声明及支持文档，答案不会发送给模型。

**不要根据留出结果再修改prompt或规则。** 若运行环境中断，先下载/恢复现有文件，不能简单删除记录再刷一遍。本格保留开始标记，重复执行不会发起第二轮。API错误停止后续请求，未返回的案例仍保留在评估分母。


In [ ]:
RUN_HOLDOUT=True #@param {type:"boolean"}
if RUN_HOLDOUT:
    assert FREE_CHECKS_OK
    assert_frozen()
    marker=PROJECT/'holdout_run_state.json'
    if marker.exists():
        print('留出运行已启动过，本格不会再次请求。现有状态：')
        print(marker.read_text())
    else:
        import getpass
        key=os.environ.get('OPENROUTER_API_KEY')
        if not key:
            try:
                from google.colab import userdata
                key=userdata.get('OPENROUTER_API_KEY')
            except Exception:key=None
        if not key:key=getpass.getpass('输入OpenRouter API key（隐藏）：').strip()
        if not key:raise ValueError('未提供key，没有调用API。')
        os.environ['OPENROUTER_API_KEY']=key
        baseline.run(PROJECT/'data/holdout_inputs.json',PROJECT/'results/holdout_baseline')
        marker.write_text(json.dumps({'status':'started','started_at_utc':datetime.now(timezone.utc).isoformat()},indent=2))
        holdout_dir=sow.run(PROJECT/'data/holdout_inputs.json',PROJECT/'results/holdout_ai',model=MODEL)
        summary=json.loads((holdout_dir/'summary.json').read_text())
        marker.write_text(json.dumps({'status':'returned','results_relative':str(holdout_dir.relative_to(PROJECT)),
                                     'cases_saved':summary['cases'],'new_api_calls':summary['new_api_calls']},indent=2))
        print('留出请求已结束。请继续生成报告，并下载备份。')
else:
    print('留出开关关闭，没有调用API。')


留出运行已启动过，本格不会再次请求。现有状态：
{
  "status": "returned",
  "results_relative": "results/holdout_ai/20261004T075011236089Z",
  "cases_saved": 5,
  "new_api_calls": 5
}


## 7 · 留出结果单独报告（免费）
只评估保存输出，不重新请求。根据第5步填写holdout人工结论后，可重新执行本格更新报告。

In [ ]:
marker=PROJECT/'holdout_run_state.json'
if not marker.exists():
    print('尚未运行留出集。')
else:
    state=json.loads(marker.read_text())
    if state.get('status')!='returned':
        print('此前运行中断。不要重发请求；先下载现有文件。',state)
    else:
        holdout_dir=PROJECT/state['results_relative']
        evaluate.report(PROJECT/'data/holdout_inputs.json',PROJECT/'labels/holdout_ground_truth.json',
            {'baseline':PROJECT/'results/holdout_baseline','ai':holdout_dir},PROJECT/'reports/holdout','holdout',PROJECT/'manual_decisions.json')
        display(Markdown((PROJECT/'reports/holdout/summary.md').read_text()))
        display(HTML((PROJECT/'reports/holdout/manual_review.html').read_text()))


# holdout 对比结果

行动/类型/问题集合匹配是分项指标，不是总体准确率。整体通过必须按固定规则人工确认。

|系统|案例数|行动匹配|类型匹配|问题集合匹配|应转人工中实际转人工|正常中转人工|API费用USD|人工完成|
|---|---:|---:|---:|---:|---|---|---|---|
|baseline|5|3/5|4/5|1/5|2/3|1/1|0.0|5/5|
|ai|5|5/5|4/5|2/5|3/3|0/1|0.0023028|5/5|

费用为每案例保留的独立响应费用，复用开发结果不是本次新支出。baseline API费用为0，未估算CPU成本。无响应案例保留在分母中。引用存在性不代表语义支持；AI引用文本由程序附上，不能称为模型自主引用准确率。

baseline 完整任务通过数：1/5

ai 完整任务通过数：2/5


## 8 · 下载这一阶段所有材料（免费）

做完开发集就可以先下载一次，做完留出后再下载。包内有baseline、冻结清单、评分程序、两套数据与答案、人工记录和所有结果。**仓库中的答案可用于评估，但不得作为模型输入。** 不包含环境变量中的API key。

In [ ]:
from google.colab import files
archive=Path('/content')/('SoW_Evaluation_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in PROJECT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and path.suffix not in {'.pyc','.env'}:
            z.write(path,str(Path(PROJECT.name)/path.relative_to(PROJECT)))
files.download(str(archive))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>